# classifying the scratch area of each picture

In [4]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

model = YOLO("best.pt")

#recognizing the car part and saving in a table
production_dir = Path("production_data")

def classify_car_part(y_center):
    if 0 <= y_center <= 0.49:
        return "Top"
    elif 0.50 <= y_center <= 0.59:
        return "Middle"
    elif 0.60 <= y_center <= 1:
        return "Bottom"
    else:
        return "Unknown"

rows = []

for img_path in sorted(production_dir.glob("*.png"), key=lambda x: int(x.stem)):
    results = model.predict(img_path, conf=0.37, verbose=False)
    result = results[0]

    if len(result.boxes) == 0:
        rows.append({
            "Image": img_path.name,
            "Scratch detected": "No",
            "Confidence": None,
            "Y coordinate": None,
            "Car part": "No scratch found"
        })
        continue

    # just in case for recognition ofmultiple bounding boxes
    best_box = max(result.boxes, key=lambda box: box.conf[0].item())

    x1, y1, x2, y2 = best_box.xyxy[0].tolist()
    img_height = result.orig_shape[0]
    y_center = ((y1 + y2) / 2) / img_height

    car_part = classify_car_part(y_center)

    rows.append({
        "Image": img_path.name,
        "Scratch detected": "Yes",
        "Confidence": round(best_box.conf[0].item(), 3),
        "Y coordinate": round(y_center, 3),
        "Car part": car_part
    })

df = pd.DataFrame(rows)

total_top_parts = (df["Car part"] == "Top").sum()
total_middle_parts = (df["Car part"] == "Middle").sum()
total_bottom_parts = (df["Car part"] == "Bottom").sum()

display(df)

print(f"Total Top Parts: {total_top_parts}")
print(f"Total Middle Parts: {total_middle_parts}")
print(f"Total Bottom Parts: {total_bottom_parts}")



,Image,Scratch detected,Confidence,Y coordinate,Car part
0,1.png,Yes,0.759,0.461,Top
1,2.png,Yes,0.682,0.552,Middle
2,3.png,Yes,0.869,0.677,Bottom
3,4.png,Yes,0.793,0.679,Bottom
4,5.png,Yes,0.716,0.438,Top
5,6.png,Yes,0.958,0.559,Middle
6,7.png,Yes,0.834,0.557,Middle
7,8.png,Yes,0.930,0.555,Middle
8,9.png,Yes,0.904,0.463,Top
9,10.png,Yes,0.900,0.437,Top


Total Top Parts: 7
Total Middle Parts: 7
Total Bottom Parts: 6


# Calculating the best result


In [ ]:
#use these variables: total_top_parts, total_middle_parts, total_bottom_parts 
